# Install Dependecies

In [1]:
%%capture
%pip install numpy
%pip install pandas
%pip install matplotlib.pyplot
%pip install python-terrier
%pip install gensim
%pip install "pyterrier-alpha[parallel]"
%pip install ipynbname
%pip install torch
%pip install transformers
%pip install accelerate
%pip install sentencepiece

In [2]:
# Load java
!curl -s "https://get.sdkman.io" | bash && source "$HOME/.sdkman/bin/sdkman-init.sh" && sdk install java 11.0.22-amzn < /dev/null

# check java and version
!which java
!java -version
!readlink -f $(which java)
!ls -la /usr/lib/jvm
!java --version
!javac --version


                                -+syyyyyyys:
                            `/yho:`       -yd.
                         `/yh/`             +m.
                       .oho.                 hy                          .`
                     .sh/`                   :N`                `-/o`  `+dyyo:.
                   .yh:`                     `M-          `-/osysoym  :hs` `-+sys:      hhyssssssssy+
                 .sh:`                       `N:          ms/-``  yy.yh-      -hy.    `.N-````````+N.
               `od/`                         `N-       -/oM-      ddd+`     `sd:     hNNm        -N:
              :do`                           .M.       dMMM-     `ms.      /d+`     `NMMs       `do
            .yy-                             :N`    ```mMMM.      -      -hy.       /MMM:       yh
          `+d+`           `:/oo/`       `-/osyh/ossssssdNMM`           .sh:         yMMN`      /m.
         -dh-           :ymNMMMMy  `-/shmNm-`:N/-.``   `.sN            /N-         `NMMy      .m/
  

# Imports

In [3]:
import itertools
import json
import os
import re
import time
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pyterrier as pt
import gensim.downloader as api
from pathlib import Path
from tqdm.auto import tqdm
import pyterrier_alpha as pta
import ipynbname

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
ROOT_DIR = ipynbname.path().parent
ROOT_DIR = Path(ROOT_DIR)
print(ROOT_DIR)

/home/tlvj/msc_datalogi/2_semester/se/notebooks


# PyTerrier - Local

In [5]:
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["JVM_PATH"] = "/usr/lib/jvm/java-11-openjdk-amd64/lib/server/libjvm.so"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

import pyterrier as pt

if not pt.java.started():
    pt.java.init()

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JVM_PATH:", os.environ["JVM_PATH"])
print("Java started:", pt.java.started())

JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
JVM_PATH: /usr/lib/jvm/java-11-openjdk-amd64/lib/server/libjvm.so
Java started: True


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


# Load Dataset

Task 7 evaluates the trained search engine on the 5,000 **unseen** test queries
(`test_queries.csv`, released on Absalon). We have no qrels for these queries, so
we only *produce* runs here — evaluation is done by the course staff. The training
queries/qrels are loaded too, only for the "5 best / 5 worst seen queries"
reflection requested in the report (Sec. 7.3, point 4).

In [6]:
base = Path.cwd() / "msc_datalogi" / "2_semester" / "se" / "project_handout"
docs          = pd.read_json(f'{base}/docs2.jsonl', lines=True, dtype={'docno': str})
train_queries = pd.read_csv(f'{base}/train_queries.csv')
train_qrels   = pd.read_csv(f'{base}/train_qrels.csv')
test_queries  = pd.read_csv(f'{base}/test_queries.csv')

print("docs.shape         :", docs.shape)
print("train_queries.shape:", train_queries.shape)
print("train_qrels.shape  :", train_qrels.shape)
print("test_queries.shape :", test_queries.shape)
test_queries.head()

docs.shape         : (151993, 2)
train_queries.shape: (10000, 2)
train_qrels.shape  : (17746, 4)
test_queries.shape : (5000, 2)


,qid,text
0,10000,where does the waikato river begin and end
1,10001,what type of gas is produced during fermentation
2,10002,why was star wars episode iv released first
3,10003,where did the oregon trail start in missouri
4,10004,who got eaten by a whale in the bible


# Load Indexes

In [7]:
index_path_none      = str((ROOT_DIR / ".." / "indexes" / "full").resolve())
index_path_stop      = str((ROOT_DIR / ".." / "indexes" / "stopwords").resolve())
index_path_stem      = str((ROOT_DIR / ".." / "indexes" / "stemming").resolve())
index_path_stop_stem = str((ROOT_DIR / ".." / "indexes" / "stop-stem").resolve())

index_none      = pt.IndexFactory.of(index_path_none)
index_stop      = pt.IndexFactory.of(index_path_stop)
index_stem      = pt.IndexFactory.of(index_path_stem)
index_stop_stem = pt.IndexFactory.of(index_path_stop_stem)

indices = {
    "stopwords": index_stop,
    "stop_stem": index_stop_stem,
    "none": index_none,
    "stem": index_stem,
}

# Preprocessing

In [8]:
# Training queries / qrels (only needed for the seen-query reflection table).
if "text" in train_queries.columns and "query" not in train_queries.columns:
    train_queries = train_queries.rename(columns={"text": "query"})
train_queries["qid"]   = train_queries["qid"].astype(str)
train_queries["query"] = train_queries["query"].astype(str)

train_qrels["qid"]   = train_qrels["qid"].astype(str)
train_qrels["docno"] = train_qrels["docno"].astype(str)
if "label" not in train_qrels.columns:
    train_qrels = train_qrels.rename(
        columns={"relevance": "label"} if "relevance" in train_qrels.columns else {"rel": "label"}
    )
train_qrels["label"] = train_qrels["label"].astype(int)

# Unseen test queries: same schema (qid, text) -> (qid, query).
if "text" in test_queries.columns and "query" not in test_queries.columns:
    test_queries = test_queries.rename(columns={"text": "query"})
test_queries["qid"]   = test_queries["qid"].astype(str)
test_queries["query"] = test_queries["query"].astype(str)

print(f"{len(test_queries)} unseen queries, qid range "
      f"{test_queries['qid'].min()}..{test_queries['qid'].max()}")
test_queries.head()

5000 unseen queries, qid range 10000..14999


,qid,query
0,10000,where does the waikato river begin and end
1,10001,what type of gas is produced during fermentation
2,10002,why was star wars episode iv released first
3,10003,where did the oregon trail start in missouri
4,10004,who got eaten by a whale in the bible


# Import tuning caches

We do **not** re-tune anything in Task 7. Every configuration we run on the unseen
queries is rebuilt from the tuning caches produced in Tasks 3, 4 and 6, so the runs
are exactly the tuned systems described in those sections.

In [33]:
cache_dir = (ROOT_DIR / ".." / "results").resolve()

bm25_cache_path   = cache_dir / "bm25_tuning_results.json"
lm_cache_path     = cache_dir / "lm_tuning_results.json"
rm3_cache_path    = cache_dir / "rm3_tuning_results.json"
rm3_lm_cache_path = cache_dir / "rm3_lm_tuning_results.json"

with open(bm25_cache_path)   as f: bm25_cache   = json.load(f)
with open(lm_cache_path)     as f: lm_cache     = json.load(f)
with open(rm3_cache_path)    as f: rm3_cache    = json.load(f)
with open(rm3_lm_cache_path) as f: rm3_lm_cache = json.load(f)

EVAL_MEASURE = "ndcg_cut_10"

# Single best (index, model) backbone from Task 3 = BM25 on stop_stem.
BACKBONE_INDEX_NAME = "stop_stem"
backbone_index      = indices[BACKBONE_INDEX_NAME]

BM25_CFG     = bm25_cache["best_configs"][BACKBONE_INDEX_NAME]["config"]   # {'k1':.., 'b':..}
LM_CFG       = lm_cache["best_configs"][BACKBONE_INDEX_NAME]["config"]     # {'c':..}
RM3_BM25_CFG = rm3_cache["best_config"]                                    # {'fb_terms':.., 'fb_docs':..}
RM3_LM_CFG   = rm3_lm_cache["best_config"]                                 # {'fb_terms':.., 'fb_docs':..}

print("BM25 backbone   :", BM25_CFG)
print("Hiemstra_LM     :", LM_CFG)
print("RM3 (BM25 final):", RM3_BM25_CFG)
print("RM3 (LM final)  :", RM3_LM_CFG)

BM25 backbone   : {'k1': 0.9, 'b': 0.6}
Hiemstra_LM     : {'c': 0.05}
RM3 (BM25 final): {'fb_terms': 50, 'fb_docs': 3}
RM3 (LM final)  : {'fb_terms': 50, 'fb_docs': 5}


# 7 Evaluation on unseen queries

We submit the **3 best configurations** by training NDCG@10, all on the single best
backbone (BM25 / stop_stem):

| Run tag        | System                            | Index     | Train NDCG@10 |
|----------------|-----------------------------------|-----------|---------------|
| `rml500LMR050` | Hiemstra_LM + RM3 (BM25 feedback) | stop_stem | 0.4580        |
| `rmb500BMR030` | BM25 + RM3                        | stop_stem | 0.4565        |
| `bms090BAS060` | BM25 baseline                     | stop_stem | 0.4516        |

Run names follow the required `{3 lower}{3 digit}{3 upper}{3 digit}` format; the
digits encode each system's key parameters as a mnemonic. Each run is written as a
top-1000 TREC file and the three files are zipped for Absalon.

## Backbone retrievers

In [34]:
# All final retrievals return up to 1000 documents (Task 7 requires top-1000).
bm25_ctrls = {"bm25.k_1": BM25_CFG["k1"], "bm25.b": BM25_CFG["b"]}

bm25_retriever = pt.terrier.Retriever(
    backbone_index, wmodel="BM25", controls=bm25_ctrls, num_results=1000,
)
print(f"Backbone: BM25/{BACKBONE_INDEX_NAME} {BM25_CFG}; LM final {LM_CFG}")

Backbone: BM25/stop_stem {'k1': 0.9, 'b': 0.6}; LM final {'c': 0.05}


## RM3 pipelines

Both RM3 systems use the single best backbone (BM25 / stop_stem) to retrieve the
top-`fb_docs` feedback documents, exactly as tuned in Task 4. They differ only in
the **final** ranker of the expanded query (BM25 vs. Hiemstra_LM).

In [35]:
def build_rm3_bm25():
    ft, fd = RM3_BM25_CFG["fb_terms"], RM3_BM25_CFG["fb_docs"]
    first  = pt.terrier.Retriever(backbone_index, wmodel="BM25", controls=bm25_ctrls, num_results=fd)
    rm3    = pt.rewrite.RM3(backbone_index, fb_terms=ft, fb_docs=fd)
    second = pt.terrier.Retriever(backbone_index, wmodel="BM25", controls=bm25_ctrls, num_results=1000)
    return first >> rm3 >> second

In [36]:
def build_rm3_lm():
    ft, fd = RM3_LM_CFG["fb_terms"], RM3_LM_CFG["fb_docs"]
    first  = pt.terrier.Retriever(backbone_index, wmodel="BM25", controls=bm25_ctrls, num_results=fd)
    rm3    = pt.rewrite.RM3(backbone_index, fb_terms=ft, fb_docs=fd)
    second = pt.terrier.Retriever(backbone_index, wmodel="Hiemstra_LM",
                                  controls={"c": LM_CFG["c"]}, num_results=1000)
    return first >> rm3 >> second

## Run configurations and TREC writer

In [37]:
runs_dir = (ROOT_DIR / ".." / "results" / "unseen_runs").resolve()
runs_dir.mkdir(parents=True, exist_ok=True)

In [38]:
def write_trec_run(results, run_tag, top_k=1000):
    """Write a results frame to results/unseen_runs/<run_tag>.txt in TREC format:
    `qid Q0 docno rank score tag`, top_k docs/query, scores in non-increasing order."""
    out_path = runs_dir / f"{run_tag}.txt"
    df = results[["qid", "docno", "score"]].copy()
    df["qid"]   = df["qid"].astype(str)
    df["docno"] = df["docno"].astype(str)
    df = df.sort_values(["qid", "score"], ascending=[True, False])
    df["rank"] = df.groupby("qid").cumcount()
    df = df[df["rank"] < top_k]
    with open(out_path, "w") as f:
        for r in df.itertuples(index=False):
            f.write(f"{r.qid} Q0 {r.docno} {r.rank} {r.score:.6f} {run_tag}\n")
    print(f"  wrote {out_path.name}: {df['qid'].nunique()} queries, {len(df)} lines")
    return out_path

In [39]:
# name -> system builder (lazy, so heavy pipelines are only built when run).
RUN_CONFIGS = [
    {"name": "rml500LMR050", "desc": "Hiemstra_LM + RM3 (BM25 feedback) / stop_stem",
     "fn": lambda: build_rm3_lm().transform(test_queries)},
    {"name": "rmb500BMR030", "desc": "BM25 + RM3 / stop_stem",
     "fn": lambda: build_rm3_bm25().transform(test_queries)},
    {"name": "bms090BAS060", "desc": "BM25 baseline / stop_stem",
     "fn": lambda: bm25_retriever.transform(test_queries)},
]

In [40]:
# Validate run names against the required format before doing any work.
NAME_RE = re.compile(r"^[a-z]{3}[0-9]{3}[A-Z]{3}[0-9]{3}$")
for cfg in RUN_CONFIGS:
    assert NAME_RE.match(cfg["name"]), f"Invalid run name: {cfg['name']}"
assert len({c["name"] for c in RUN_CONFIGS}) == len(RUN_CONFIGS), "Run names must be unique"
assert 3 <= len(RUN_CONFIGS) <= 5, "Submit between 3 and 5 runs"
print("Run names OK:", [c["name"] for c in RUN_CONFIGS])

Run names OK: ['rml500LMR050', 'rmb500BMR030', 'bms090BAS060']


## Generate the runs (cached as .txt)

Each run is written once; on re-run an existing `.txt` is kept (pass `force=True`
to regenerate). RM3 retrieval over 5,000 queries is the slow part.

In [41]:
def generate_run(cfg, force=False):
    out_path = runs_dir / f"{cfg['name']}.txt"
    if out_path.exists() and not force:
        print(f"[skip] {cfg['name']} already exists ({cfg['desc']})")
        return out_path
    print(f"[run ] {cfg['name']}: {cfg['desc']}")
    t0 = time.time()
    results = cfg["fn"]()
    write_trec_run(results, cfg["name"])
    print(f"       done in {time.time()-t0:.1f}s")
    return out_path

In [42]:
run_paths = [generate_run(cfg) for cfg in RUN_CONFIGS]
run_paths

[run ] rml500LMR050: Hiemstra_LM + RM3 (BM25 feedback) / stop_stem
  wrote rml500LMR050.txt: 5000 queries, 5000000 lines
       done in 163.7s
[run ] rmb500BMR030: BM25 + RM3 / stop_stem
  wrote rmb500BMR030.txt: 5000 queries, 5000000 lines
       done in 159.6s
[run ] bms090BAS060: BM25 baseline / stop_stem
  wrote bms090BAS060.txt: 5000 queries, 4978950 lines
       done in 74.9s


[PosixPath('/home/tlvj/msc_datalogi/2_semester/se/results/unseen_runs/rml500LMR050.txt'),
 PosixPath('/home/tlvj/msc_datalogi/2_semester/se/results/unseen_runs/rmb500BMR030.txt'),
 PosixPath('/home/tlvj/msc_datalogi/2_semester/se/results/unseen_runs/bms090BAS060.txt')]

## Zip for Absalon submission

In [43]:
submission_zip = runs_dir / "se_unseen_runs.zip"
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for cfg in RUN_CONFIGS:
        p = runs_dir / f"{cfg['name']}.txt"
        zf.write(p, arcname=p.name)   # flat archive: just the .txt files

print(f"Wrote {submission_zip}")
with zipfile.ZipFile(submission_zip) as zf:
    print("Contents:", zf.namelist())

Wrote /home/tlvj/msc_datalogi/2_semester/se/results/unseen_runs/se_unseen_runs.zip
Contents: ['rml500LMR050.txt', 'rmb500BMR030.txt', 'bms090BAS060.txt']


## Sanity check

Confirm the TREC format, that every run covers all 5,000 unseen queries, and that
no run exceeds 1,000 docs/query.

In [44]:
n_test = test_queries["qid"].nunique()
for cfg in RUN_CONFIGS:
    p = runs_dir / f"{cfg['name']}.txt"
    df = pd.read_csv(p, sep=r"\s+", header=None,
                     names=["qid", "Q0", "docno", "rank", "score", "tag"],
                     dtype={"qid": str, "docno": str})
    per_q = df.groupby("qid").size()
    ok_cols = (df["Q0"] == "Q0").all() and (df["tag"] == cfg["name"]).all()
    ok_desc = df.groupby("qid")["score"].apply(lambda s: s.is_monotonic_decreasing).all()
    print(f"{cfg['name']}: queries={df['qid'].nunique()}/{n_test}  "
          f"max_docs/q={per_q.max()}  Q0&tag_ok={ok_cols}  scores_desc={ok_desc}")

print("\nExample lines from", RUN_CONFIGS[0]["name"])
print((runs_dir / f"{RUN_CONFIGS[0]['name']}.txt").read_text().splitlines()[:5])

rml500LMR050: queries=5000/5000  max_docs/q=1000  Q0&tag_ok=True  scores_desc=True
rmb500BMR030: queries=5000/5000  max_docs/q=1000  Q0&tag_ok=True  scores_desc=True
bms090BAS060: queries=5000/5000  max_docs/q=1000  Q0&tag_ok=True  scores_desc=True

Example lines from rml500LMR050
['10000 Q0 D83159 0 20.338746 rml500LMR050', '10000 Q0 D106289 1 17.048895 rml500LMR050', '10000 Q0 D121323 2 14.251997 rml500LMR050', '10000 Q0 D62984 3 13.392619 rml500LMR050', '10000 Q0 D112862 4 10.808949 rml500LMR050']
